# Exp 8: confinement curve

Same projection as Exp 7, but only a share p of the weights is kept inside its INT4 cell (p = 0, 0.25, 0.5, 0.75, 0.9, 1, nested random masks). A second family confines everything except the weights that moved most in the unconstrained run.

For every variant: how much it forgets in FP32 vs how much INT4 undoes. Cora and CiteSeer, seeds 70-74, 400 steps.

CPU, 45-75 min. The output folder has a config fingerprint so a resumed run cannot mix settings.

In [ ]:
# 0. Install (Colab and Kaggle already have PyTorch)
!pip install -q torch_geometric

## 1. Configuration

In [ ]:
import os, json, time, random, shutil, datetime, itertools
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torch_geometric
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.utils import subgraph
from torch_geometric.nn import GCNConv
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from sklearn.metrics import roc_auc_score
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

SMOKE_TEST = False          # True = tiny crash test (results are NOT meaningful)
SAVE_TO_DRIVE = False       # Colab only

if SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/exp8_confinement_curve'
else:
    OUT_DIR = 'exp8_confinement_curve' if not SMOKE_TEST else 'exp8_smoke_test'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CFG = dict(
    datasets=['CiteSeer', 'Cora'], primary_dataset='CiteSeer',
    hidden=128, dropout=0.5, lr_train=0.01, weight_decay=5e-4, epochs=200,         # frozen since Experiment 5
    train_frac=0.10, delete_frac=0.10,
    ft_lr=0.005, ft_dose=100,                                                      # Experiment 6 FT, unchanged (reference)
    variant_steps=400,                                                             # fixed, no selection
    rc_shares=[0.0, 0.25, 0.5, 0.75, 0.9, 1.0], tf_free=[0.01, 0.05, 0.10, 0.25],
    cell_bits=4, cell_margin=0.05,
    n_retrain=5, seeds=[70, 71, 72, 73, 74],
    bits=[8, 4, 3], schemes=['per_channel'], scale_modes=['recomputed', 'frozen_original'],
    noise_draws=3, min_seeds_agree=4, utility_tolerance=0.02, collapse_drop=0.05,
    existence_min_reversal=0.10, unstable_rms=0.01,
)
if SMOKE_TEST:
    CFG.update(epochs=20, ft_dose=20, variant_steps=30, seeds=[70, 71], n_retrain=3, noise_draws=1, min_seeds_agree=1,
               rc_shares=[0.0, 0.25, 0.5, 1.0], tf_free=[0.05, 0.10, 0.25])
    print('SMOKE TEST: tiny settings, results are not meaningful.')

def log(msg):
    line = f'[{datetime.datetime.now():%H:%M:%S}] {msg}'
    print(line, flush=True)
    with open(f'{OUT_DIR}/progress.log', 'a') as f: f.write(line + '\n')

ENV = dict(device=str(DEVICE), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           torch=torch.__version__, torch_geometric=torch_geometric.__version__, smoke_test=SMOKE_TEST,
           started=f'{datetime.datetime.now():%Y-%m-%d %H:%M:%S}')
# --- compatibility check BEFORE anything is written into the results folder ---
import hashlib
FINGERPRINT = hashlib.sha256(json.dumps(CFG, sort_keys=True).encode()).hexdigest()[:16]
_fp_file = f'{OUT_DIR}/config_fingerprint.json'
if os.path.exists(_fp_file):
    _old = json.load(open(_fp_file))['fingerprint']
    if _old != FINGERPRINT:
        raise RuntimeError(f'Results folder {OUT_DIR} was created with a different configuration ({_old} vs {FINGERPRINT}). '
                           'Nothing was written. Use a new results folder or restore the original settings.')
elif os.path.isdir(OUT_DIR) and any(os.scandir(OUT_DIR)):
    raise RuntimeError(f'Results folder {OUT_DIR} has files but no configuration fingerprint. Nothing was written. Use a new folder.')
# --- check passed: only now create folders and write files ---
for sub in ['fp32', 'quant', 'checkpoints']: os.makedirs(f'{OUT_DIR}/{sub}', exist_ok=True)
if not os.path.exists(_fp_file): json.dump(dict(fingerprint=FINGERPRINT, config=CFG), open(_fp_file, 'w'), indent=2)
json.dump(CFG, open(f'{OUT_DIR}/config.json', 'w'), indent=2); json.dump(ENV, open(f'{OUT_DIR}/environment.json', 'w'), indent=2)
RC = [f'RC-p{p:.2f}' for p in CFG['rc_shares']]; TF = [f'TF-f{f:.2f}' for f in CFG['tf_free']]
VARIANTS = RC + TF; FTN = f"FT dose={CFG['ft_dose']}"; TEST_NAMES = [FTN] + VARIANTS
log(f'Device: {DEVICE} | results folder: {OUT_DIR} | config fingerprint {FINGERPRINT} | variants: {VARIANTS}')

## 2. Pre-registration (written and saved before any result of this experiment)

In [ ]:
PREREG = f'''# Experiment 8: confinement curve. Pre-registration
Saved: {datetime.datetime.now():%Y-%m-%d %H:%M:%S}, before any result of this experiment. Device: {ENV['device']} ({ENV['gpu']}).

## Background (unchanged by this experiment)
Experiment 6: fine-tuning unlearning (lr 0.005, 100 steps) was PRESERVED in all 36 quantization configurations.
Experiment 7 (pre-registered verdict: positive control not constructible): fine-tuning confined to the INT4 per-channel cells
of the Original was undone by INT4 (reversal fraction 0.97, CI 0.95-1.00) but reached only about 60 % of the forgetting
needed (distance to Retrain 0.102 vs band 0.017; flat from 100 to 800 steps); half confinement: reversal 0.34, distance 0.049.
Two analysis lessons from Experiment 7 are applied here from the start (see Analysis).

## Settings
Datasets {CFG['datasets']} (primary {CFG['primary_dataset']}); model, training, split and deletion exactly as in Experiments 6-7.
Seeds {CFG['seeds']} (fresh). {CFG['n_retrain']} Retrains per seed; Retrain-1 is the reference.

## Models per seed (FP32 checkpoints saved; quantization only uses copies)
Original; Retrain-1..{CFG['n_retrain']}; FT = Experiment 6 fine-tuning ({CFG['ft_dose']} steps, unconstrained, reference).
Variants: FT objective (retained labels, reduced graph, Adam lr {CFG['ft_lr']}, weight decay {CFG['weight_decay']}, dropout on),
{CFG['variant_steps']} steps (fixed in advance, no selection), projection after every step as in Experiment 7: biases frozen;
per-row maximum |w| held fixed and other entries kept inside (-max, +max) of their row; confined entries kept in
[(c - 0.5 + {CFG['cell_margin']}) s, (c + 0.5 - {CFG['cell_margin']}) s], s = row max|w_O| / 7 (INT4 per-channel cell).
* RC-p (random confinement): a random share p of entries confined, p in {CFG['rc_shares']}. The masks are NESTED: one random
  score per entry per seed, entry confined if score < p, so raising p only adds constraints. RC-p0.00 = Experiment 7's PC-none
  at {CFG['variant_steps']} steps; RC-p1.00 = PC-full (its INT4 per-channel codes equal the quantized Original's; asserted).
* TF-f (targeted freeing): every entry confined except the share f with the largest |W_RC-p0.00 - W_O| / s in its matrix
  (same seed), f in {CFG['tf_free']}.

## Measurements
Identical to Experiments 6-7 (membership AUC on deleted vs held-out nodes, calibrated attacks with recalibrated and FP32
thresholds, utility, update survival, node-level raw shifts). Quantization: INT{CFG['bits']} per-channel, recomputed and
frozen-Original scales, matched-noise controls ({CFG['noise_draws']} draws). Per-tensor is not repeated (covered in Experiments 6-7).

## Quantities per variant (paired over seeds, 95 % t-intervals, k/5 counts)
Forgetting achieved phi = 1 - dist_FP32(variant) / dist_FP32(Original), dist = |AUC - AUC(Retrain-1)| (1 = matches Retrain-1).
Reversal fraction rho = (gap_Q(v) - gap_FP32(v)) / (gap_FP32(O) - gap_FP32(v)), gap = AUC - AUC(Retrain-1) at the same precision.
Denominator diagnostic: on a seed, rho is flagged unstable if |gap_FP32(O) - gap_FP32(v)| < band_FP32 (the variant's
forgetting is within Retrain-to-Retrain variation). A variant with any unstable seed, or with mean phi < 0.2, has its rho
reported but flagged, and is excluded from E1 and E2.
Boundary crossings (direct mechanism measure): share of weight entries whose INT4 per-channel integer code differs between the
quantized variant and the quantized Original, overall and per layer (alongside the share of updates above half a bin).

## FP32 gate (per variant, as in Experiment 6)
Improvement CI above 0, >= {CFG['min_seeds_agree']} seeds improving, mean dist <= band_FP32 (5-Retrain band, descriptive tolerance),
utility >= Original - {CFG['utility_tolerance']}, Original fails the band.

## Analysis rules (two refinements carried over from Experiment 7, declared in advance)
1. The Experiment 6 verdicts (REVERSAL / PRESERVED / ...) are only assigned to variants that pass the FP32 gate. For variants
   that do not pass, only rho (with CI) is reported, because "distance above band" is already true before quantization.
2. Deleted-node specificity of a quantization shift is judged by the raw confidence shifts: specific if the CI of
   (mean shift on deleted nodes) - (mean shift on held-out nodes) is above 0. Normalised node recovery is reported only.

## Pre-registered questions
E1 (primary, existence; {CFG['primary_dataset']}, INT4 per-channel recomputed): is there a variant that passes the FP32 gate, receives the
   formal Experiment 6 REVERSAL verdict (which includes the matched-noise criterion; thresholds unchanged), has no unstable rho,
   AND has rho with lower 95 % bound > {CFG['existence_min_reversal']}? The continuous rho is reported separately for all variants. Answer "yes: fragile complete forgetting exists" or
   "no variant both reached Retrain and was undone by INT4". Same question for Cora (secondary).
E2 (trade-off, descriptive): Spearman correlation between variant means of phi and rho, computed separately for the random
   (RC) and targeted (TF) families (eligible variants only). The variants share Originals, splits and Retrains, so no ordinary
   p-value is reported. Uncertainty: (a) seed-level analysis, the same correlation within each seed (mean and k/5 seeds negative);
   (b) cluster bootstrap over seeds (2000 resamples of whole seeds, preserving within-seed dependence), percentile 95 % interval.
   Expected sign: negative.
E3 (targeted): smallest f in the TF family that passes the FP32 gate, and its rho; i.e. how few weights need to cross a
   rounding boundary for forgetting that survives INT4.

## Implementation corrections after independent review (made before any full run)
Nested random masks; family-separate descriptive Spearman with seed-level and cluster-bootstrap uncertainty instead of an
ordinary p-value; integer-code change shares; rho denominator diagnostic; a configuration fingerprint that stops the notebook
from reusing results or checkpoints created with different settings. Second review: the fingerprint is checked before
anything is written to the results folder (a rejected resume leaves every existing file unchanged), and E1 requires the
formal REVERSAL verdict including the matched-noise criterion. Interpretation note: the TF family uses oracle information
(the unconstrained fine-tuning run) to choose weights; it is a mechanism probe, not a deployable unlearning method. The scientific design (seeds, 400 steps, FP32 gate,
variants, quantization settings, E1 and E3) is unchanged.

## Scope
Conclusions concern this measured membership attack and fine-tuning-type updates optimised with projected Adam; they do not
show that no within-cell solution exists. Variants are analysis tools, not proposed unlearning methods.
'''
path = f'{OUT_DIR}/exp8_preregistration.md'
if os.path.exists(path):
    log('Pre-registration already saved earlier: keeping the original file unchanged.')
else:
    open(path, 'w').write(PREREG); log('Pre-registration saved.')
print(open(path).read())

## 3. Model (identical to Experiment 6)

In [ ]:
def norm_adj(edge_index, n):
    ei, w = gcn_norm(edge_index, None, n, add_self_loops=True)
    return torch.sparse_coo_tensor(torch.stack([ei[1], ei[0]]), w, (n, n)).coalesce().to(DEVICE)

class Layer(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.lin = nn.Linear(i, o, bias=False); self.bias = nn.Parameter(torch.zeros(o))
        nn.init.xavier_uniform_(self.lin.weight)
    def forward(self, x, A):
        xw = torch.sparse.mm(x, self.lin.weight.t()) if x.is_sparse else x @ self.lin.weight.t()
        return torch.sparse.mm(A, xw) + self.bias

class GCN(nn.Module):
    def __init__(self, i, h, o, p):
        super().__init__(); self.c1, self.c2, self.p = Layer(i, h), Layer(h, o), p
    def forward(self, xs, A):
        if self.training and self.p > 0:
            xs = torch.sparse_coo_tensor(xs.indices(), F.dropout(xs.values(), self.p, True), xs.shape)
        h = F.relu(self.c1(xs, A)); h = F.dropout(h, self.p, self.training)
        return self.c2(h, A)

# parameter order: c1.bias, c1.lin.weight (2-D), c2.bias, c2.lin.weight (2-D); weights are always selected by shape.
_d = Planetoid('data', 'Cora', transform=T.NormalizeFeatures())[0]
_m = GCN(_d.num_features, 128, 7, 0.5).to(DEVICE).eval()
assert [p.dim() for p in _m.parameters()] == [1, 2, 1, 2]
_r1, _r2 = GCNConv(_d.num_features, 128).to(DEVICE), GCNConv(128, 7).to(DEVICE)
with torch.no_grad():
    _m.c1.bias.add_(0.1)
    _r1.lin.weight.copy_(_m.c1.lin.weight); _r1.bias.copy_(_m.c1.bias); _r2.lin.weight.copy_(_m.c2.lin.weight); _r2.bias.copy_(_m.c2.bias)
    diff = (_m(_d.x.to_sparse().to(DEVICE), norm_adj(_d.edge_index, _d.num_nodes))
            - _r2(F.relu(_r1(_d.x.to(DEVICE), _d.edge_index.to(DEVICE))), _d.edge_index.to(DEVICE))).abs().max().item()
log(f'Equivalence check vs PyG GCNConv: max |difference| = {diff:.2e}')
assert diff < 1e-4, 'implementation does not match GCNConv'

## 4. Quantizer, matched noise, measurements (identical to the Experiment 6 quantization stage)

In [ ]:
def quantize(W, bits, scheme, scale_src=None):
    qmax = 2 ** (bits - 1) - 1; out, scales, clipped, total = [], [], 0, 0
    src = W if scale_src is None else scale_src
    for w, s0 in zip(W, src):
        if w.dim() != 2: out.append(w.clone()); scales.append(None); continue
        s = (s0.abs().amax(1, keepdim=True) if scheme == 'per_channel' else s0.abs().max()) / qmax
        s = torch.clamp(s, min=1e-12).expand_as(w)
        q = torch.round(w / s); clipped += int((q.abs() > qmax).sum()); total += q.numel()
        out.append(torch.clamp(q, -qmax, qmax) * s); scales.append(s)
    return out, scales, clipped / total

def matched_noise(W, Wq, scheme, gen):
    out = []
    for w, wq in zip(W, Wq):
        if w.dim() != 2: out.append(w.clone()); continue
        e = (wq - w).cpu()
        if scheme == 'per_channel':
            idx = torch.argsort(torch.rand(e.shape, generator=gen), dim=1); e = torch.gather(e, 1, idx)
        else:
            e = e.flatten()[torch.randperm(e.numel(), generator=gen)].view_as(e)
        out.append(w + e.to(w.device))
    return out

def flat2d(W): return torch.cat([w.flatten() for w in W if w.dim() == 2])
def cos(a, b): return float((a @ b) / (a.norm() * b.norm() + 1e-30))

_w = [torch.tensor([[0.70, -0.35, 0.10], [0.07, 0.02, -0.01]]), torch.zeros(2)]
assert torch.allclose(quantize(_w, 4, 'per_channel')[0][0], torch.tensor([[0.7, -0.4, 0.1], [0.07, 0.02, -0.01]]), atol=1e-6)
assert torch.allclose(quantize(_w, 4, 'per_tensor')[0][0], torch.tensor([[0.7, -0.4, 0.1], [0.1, 0.0, 0.0]]), atol=1e-6)
_q, _s, _c = quantize([_w[0] * 2, _w[1]], 4, 'per_channel', scale_src=_w)
assert abs(_c - 2 / 6) < 1e-9 and torch.allclose(_q[0][0], torch.tensor([0.7, -0.7, 0.2]), atol=1e-6)
_n = matched_noise(_w, quantize(_w, 4, 'per_channel')[0], 'per_channel', torch.Generator().manual_seed(0))
assert torch.allclose((quantize(_w, 4, 'per_channel')[0][0] - _w[0]).sort(1).values, (_n[0] - _w[0]).sort(1).values, atol=1e-7)
log('Quantizer and matched-noise self-checks passed.')

_DATA, _PROBE = {}, {}
def get_data(name):
    if name not in _DATA:
        d = Planetoid('data', name, transform=T.NormalizeFeatures())[0]
        _DATA[name] = dict(d=d, xs=d.x.to_sparse().to(DEVICE), y=d.y.to(DEVICE), y_np=d.y.numpy(),
                           A_full=norm_adj(d.edge_index, d.num_nodes), C=int(d.y.max()) + 1)
    return _DATA[name]

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def make_split(ds, seed):
    G = get_data(ds); d = G['d']; N = d.num_nodes
    rng = np.random.default_rng(seed)
    perm = rng.permutation(N); n_tr = int(CFG['train_frac'] * N)
    tr, held = np.sort(perm[:n_tr]), np.sort(perm[n_tr:])
    D = np.sort(rng.choice(tr, int(CFG['delete_frac'] * len(tr)), replace=False)); retained = np.setdiff1d(tr, D)
    hp = rng.permutation(held); H_cal, H_eval = hp[: len(hp) // 2], hp[len(hp) // 2:]
    R_cal = rng.choice(retained, size=min(len(retained), len(H_cal)), replace=False)
    return dict(train=tr, held=held, D=D, retained=retained, H_cal=H_cal, H_eval=H_eval, R_cal=R_cal)

def make_ctx(ds, sp):
    G = get_data(ds); N = G['d'].num_nodes
    keep = torch.ones(N, dtype=torch.bool); keep[torch.from_numpy(sp['D'])] = False
    ei_red, _ = subgraph(keep, G['d'].edge_index, num_nodes=N)
    if ds not in _PROBE: _PROBE[ds] = GCN(G['d'].num_features, CFG['hidden'], G['C'], CFG['dropout']).to(DEVICE)
    return dict(ds=ds, G=G, sp=sp, A_red=norm_adj(ei_red, N), N=N)

def calibrate(cr, ch):
    cand = np.unique(np.r_[cr, ch]); sr, sh = np.sort(cr), np.sort(ch)
    tpr = 1 - np.searchsorted(sr, cand, 'left') / len(sr); tnr = np.searchsorted(sh, cand, 'left') / len(sh)
    return float(cand[int(np.argmax((tpr + tnr) / 2))])

@torch.no_grad()
def measure(ctx, W, thr_fp32=None):
    G, sp, probe = ctx['G'], ctx['sp'], _PROBE[ctx['ds']]
    for p, w in zip(probe.parameters(), W): p.data = w.clone()
    probe.eval(); y = G['y_np']; N = ctx['N']
    conf = F.softmax(probe(G['xs'], G['A_full']), 1).cpu().numpy()[np.arange(N), y]
    pred_red = probe(G['xs'], ctx['A_red']).argmax(1).cpu().numpy()
    D, He, ret = sp['D'], sp['H_eval'], sp['retained']
    auc = roc_auc_score(np.r_[np.ones(len(D)), np.zeros(len(He))], np.r_[conf[D], conf[He]])
    thr = calibrate(conf[sp['R_cal']], conf[sp['H_cal']]); t2 = thr if thr_fp32 is None else thr_fp32
    m = dict(auc=auc, adv=2 * abs(auc - 0.5), threshold=thr,
             tpr_minus_fpr_recal=(conf[D] >= thr).mean() - (conf[He] >= thr).mean(),
             tpr_minus_fpr_fp32_threshold=(conf[D] >= t2).mean() - (conf[He] >= t2).mean(),
             mean_conf_D=float(conf[D].mean()), mean_conf_Heval=float(conf[He].mean()),
             auc_retained_vs_Heval=roc_auc_score(np.r_[np.ones(len(ret)), np.zeros(len(He))], np.r_[conf[ret], conf[He]]),
             test_acc=float((pred_red[sp['held']] == y[sp['held']]).mean()), acc_deleted=float((pred_red[D] == y[D]).mean()))
    return m, conf

def node_recovery(sp, conf_new, conf_ft, conf_o):
    out = {}
    for name, idx in [('D', sp['D']), ('Heval', sp['H_eval'])]:
        t, dlt = conf_o[idx] - conf_ft[idx], conf_new[idx] - conf_ft[idx]; rms = float(np.sqrt(np.mean(t ** 2)))
        out.update({f'node_recovery_{name}': float(dlt @ t / (t @ t + 1e-30)), f'raw_shift_{name}': float(dlt.mean()),
                    f'raw_target_gap_{name}': float(t.mean()), f'target_rms_{name}': rms, f'unstable_{name}': bool(rms < CFG['unstable_rms'])})
    out['raw_shift_D_minus_Heval'] = out['raw_shift_D'] - out['raw_shift_Heval']
    return out

def ci95(x):
    x = np.asarray(x, float); n = len(x); m = x.mean()
    if n < 2: return m, np.nan, np.nan
    h = stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n); return m, m - h, m + h

def retrain_band(auc_by_seed):
    cols = [c for c in auc_by_seed.columns if c.startswith('Retrain-')]
    sq = [(auc_by_seed[a] - auc_by_seed[b]) ** 2 for a, b in itertools.combinations(cols, 2)]
    return float(2 * np.sqrt(pd.concat(sq).mean()))

## 5. Confinement masks and projection
`bounds_from_mask` turns a boolean mask (True = confined to its INT4 cell) into projection bounds. The self-check verifies that
a fully confined model pushed to its cell edges quantizes exactly like the Original, and that freed entries can leave their cell.

In [ ]:
QMAX_CELL = 2 ** (CFG['cell_bits'] - 1) - 1
def cell_info(w0):
    rmax = w0.abs().amax(1, keepdim=True); s = rmax / QMAX_CELL; return rmax, s, torch.round(w0 / s)

def bounds_from_mask(W0, masks):
    # masks: list aligned with W0 (None for biases); returns (lo, hi) per parameter; biases frozen
    mg = CFG['cell_margin']; B = []
    for w0, cm in zip(W0, masks):
        if w0.dim() != 2: B.append((w0.clone(), w0.clone())); continue
        rmax, s, c = cell_info(w0); inner = rmax * (1 - 1e-6)
        lo, hi = (-inner).expand_as(w0).clone(), inner.expand_as(w0).clone()
        lo = torch.where(cm, torch.maximum(lo, (c - 0.5 + mg) * s), lo); hi = torch.where(cm, torch.minimum(hi, (c + 0.5 - mg) * s), hi)
        am = torch.zeros_like(w0, dtype=torch.bool); am[torch.arange(w0.shape[0]), w0.abs().argmax(1)] = True
        lo = torch.where(am, w0, lo); hi = torch.where(am, w0, hi)
        B.append((lo, hi))
    return B

def random_scores(W0, gen):
    # one fixed random score per entry per seed; mask(p) = score < p is nested in p (p = 1 confines everything)
    return [None if w.dim() != 2 else torch.rand(w.shape, generator=gen).to(w.device) for w in W0]

def nested_masks(scores, share):
    return [None if sc is None else (sc < share) if share < 1 else torch.ones_like(sc, dtype=torch.bool) for sc in scores]

def topfree_masks(W0, W_free, f):
    out = []
    for w0, w1 in zip(W0, W_free):
        if w0.dim() != 2: out.append(None); continue
        _, s, _ = cell_info(w0); rel = ((w1 - w0).abs() / s).flatten()
        k = max(1, int(round(f * rel.numel()))); thr = torch.topk(rel, k).values.min()
        out.append(~((w1 - w0).abs() / s >= thr))           # confined = NOT among the top-f movers
    return out

@torch.no_grad()
def project(model, B):
    for p, (lo, hi) in zip(model.parameters(), B): p.data = torch.minimum(torch.maximum(p.data, lo), hi)

def same_q_pc(Wa, Wb, mode, bits):
    srcw = Wb if mode == 'frozen_original' else None
    qa = quantize(Wa, bits, 'per_channel', srcw)[0]; qb = quantize(Wb, bits, 'per_channel')[0]
    return all(torch.equal(a, b) for a, b in zip(qa, qb) if a.dim() == 2)

_g = torch.Generator().manual_seed(1); _W0 = [torch.zeros(16), torch.randn(16, 40, generator=_g) * 0.1]
_sc = random_scores(_W0, _g); _ms = [nested_masks(_sc, p)[1] for p in [0.0, 0.25, 0.5, 0.75, 0.9, 1.0]]
assert all(bool((a <= b).all()) for a, b in zip(_ms[:-1], _ms[1:])), 'masks not nested'
assert not _ms[0].any() and _ms[-1].all()
_B = bounds_from_mask(_W0, nested_masks(_sc, 1.0))
_Wx = [torch.where(torch.rand(lo.shape, generator=_g) < 0.5, lo, hi) for lo, hi in _B]
assert same_q_pc(_Wx, _W0, 'recomputed', 4) and same_q_pc(_Wx, _W0, 'frozen_original', 4), 'cell construction broken'
_Wf = [_W0[0], _W0[1] + 0.3 * torch.randn(16, 40, generator=_g) * 0.1]
_m = topfree_masks(_W0, _Wf, 0.10); assert abs((~_m[1]).float().mean().item() - 0.10) < 0.02, 'top-free share wrong'
_B2 = bounds_from_mask(_W0, _m); _am = torch.zeros_like(_m[1]); _am[torch.arange(16), _W0[1].abs().argmax(1)] = True
_free = (~_m[1]) & ~_am; _bin = (_W0[1].abs().amax(1, keepdim=True) / 7).expand_as(_W0[1])
assert bool((_B2[1][1] - _B2[1][0])[_free].gt(_bin[_free]).all()), 'freed entries cannot leave their cell'
assert bool((_B2[1][1] - _B2[1][0])[_m[1] & ~_am].le(_bin[_m[1] & ~_am]).all()), 'confined entries wider than one cell'
log('Confinement self-checks passed.')

## 6. Training one seed (all models) and saving the exact FP32 checkpoint

In [ ]:
def ck_path(ds, seed): return f'{OUT_DIR}/checkpoints/{ds}_seed{seed}.pt'

def train_seed(ds, seed):
    if os.path.exists(ck_path(ds, seed)):
        fpk = torch.load(ck_path(ds, seed), weights_only=False).get('fingerprint')
        if fpk != FINGERPRINT: raise RuntimeError(f'Checkpoint {ck_path(ds, seed)} has fingerprint {fpk}, expected {FINGERPRINT}: not reused.')
        log(f'{ds} seed {seed}: FP32 checkpoint exists (fingerprint matches), not retraining'); return
    t0 = time.time(); G = get_data(ds); d, xs, Y, A_full = G['d'], G['xs'], G['y'], G['A_full']
    sp = make_split(ds, seed); ctx = make_ctx(ds, sp); A_red = ctx['A_red']; N = d.num_nodes
    m_full = torch.zeros(N, dtype=torch.bool, device=DEVICE); m_full[torch.from_numpy(sp['train'])] = True
    m_red = torch.zeros(N, dtype=torch.bool, device=DEVICE); m_red[torch.from_numpy(sp['retained'])] = True
    def train(A, mask, init_seed):
        set_seed(init_seed); m = GCN(d.num_features, CFG['hidden'], G['C'], CFG['dropout']).to(DEVICE)
        opt = torch.optim.Adam(m.parameters(), lr=CFG['lr_train'], weight_decay=CFG['weight_decay'])
        for _ in range(CFG['epochs']):
            m.train(); opt.zero_grad(); F.cross_entropy(m(xs, A)[mask], Y[mask]).backward(); opt.step()
        return [p.detach().clone() for p in m.parameters()]
    W = {'Original': train(A_full, m_full, 1000 + seed)}
    for k in range(1, CFG['n_retrain'] + 1): W[f'Retrain-{k}'] = train(A_red, m_red, 2000 + 1000 * (k - 1) + seed)
    def finetune(steps, B=None):
        set_seed(5000 + seed + int(CFG['ft_lr'] * 1e4))      # same dropout stream as Experiments 6-7
        m = GCN(d.num_features, CFG['hidden'], G['C'], CFG['dropout']).to(DEVICE)
        for p, w in zip(m.parameters(), W['Original']): p.data = w.clone()
        opt = torch.optim.Adam(m.parameters(), lr=CFG['ft_lr'], weight_decay=CFG['weight_decay'])
        for _ in range(steps):
            m.train(); opt.zero_grad(); F.cross_entropy(m(xs, A_red)[m_red], Y[m_red]).backward(); opt.step()
            if B is not None: project(m, B)
        return [p.detach().clone() for p in m.parameters()]
    W[FTN] = finetune(CFG['ft_dose'])
    scores = random_scores(W['Original'], torch.Generator().manual_seed(7000 + 10 * seed))     # one ordering per seed
    for i, p in enumerate(CFG['rc_shares']):
        W[RC[i]] = finetune(CFG['variant_steps'], bounds_from_mask(W['Original'], nested_masks(scores, p)))
    if 1.0 in CFG['rc_shares']:
        for mode in ['recomputed', 'frozen_original']:
            assert same_q_pc(W['RC-p1.00'], W['Original'], mode, CFG['cell_bits']), f'RC-p1.00 not identical after INT4 ({mode})'
    for i, f in enumerate(CFG['tf_free']):
        masks = topfree_masks(W['Original'], W['RC-p0.00'], f)
        W[TF[i]] = finetune(CFG['variant_steps'], bounds_from_mask(W['Original'], masks))
    torch.save(dict(weights={k: [w.cpu() for w in v] for k, v in W.items()}, split=sp, seed=seed, env=ENV,
                    fingerprint=FINGERPRINT), ck_path(ds, seed))
    log(f'{ds} seed {seed}: trained and saved checkpoint in {time.time()-t0:.0f}s (RC-p1.00 INT4 identity asserted)')

def load_ck(ds, seed):
    ck = torch.load(ck_path(ds, seed), weights_only=False)
    assert ck.get('fingerprint') == FINGERPRINT, f'checkpoint {ds} seed {seed} was made with a different configuration'
    return {k: [w.to(DEVICE) for w in v] for k, v in ck['weights'].items()}, ck['split']

def eval_fp32(ds, seed):
    f = f'{OUT_DIR}/fp32/{ds}_seed{seed}.csv'
    if os.path.exists(f): return
    W, sp = load_ck(ds, seed); ctx = make_ctx(ds, sp); rows = []
    for name, Wm in W.items():
        m, _ = measure(ctx, Wm)
        rows.append(dict(dataset=ds, seed=seed, model=name, update_norm=float((flat2d(Wm) - flat2d(W['Original'])).norm()), **m))
    pd.DataFrame(rows).to_csv(f, index=False)

for ds in CFG['datasets']:
    for seed in CFG['seeds']:
        train_seed(ds, seed); eval_fp32(ds, seed)
log('FP32 phase finished.')

## 7. FP32 gate and forgetting achieved (saved before quantization)

In [ ]:
GATE_PATH = f'{OUT_DIR}/fp32_gate.json'
if os.path.exists(GATE_PATH):
    GATE = json.load(open(GATE_PATH)); log('FP32 gate already decided earlier: NOT recomputed.')
else:
    GATE = {}; k = CFG['min_seeds_agree']
    for ds in CFG['datasets']:
        t = pd.concat([pd.read_csv(f'{OUT_DIR}/fp32/{ds}_seed{s}.csv') for s in CFG['seeds']])
        a = t.pivot_table(index='seed', columns='model', values='auc'); u = t.pivot_table(index='seed', columns='model', values='test_acc')
        band = retrain_band(a); dist_o = (a['Original'] - a['Retrain-1']).abs(); GATE[ds] = dict(band_fp32=band, dist_original=float(dist_o.mean()), models={})
        for n in TEST_NAMES:
            dist = (a[n] - a['Retrain-1']).abs(); imp = dist_o - dist; m, lo, hi = ci95(imp); phi = ci95(1 - dist / dist_o)
            c = dict(ci_above_0=bool(lo > 0), improves_on_enough_seeds=bool((imp > 0).sum() >= k), within_band=bool(dist.mean() <= band),
                     utility_ok=bool(u[n].mean() >= u['Original'].mean() - CFG['utility_tolerance']),
                     original_fails_band=bool(dist_o.mean() > band and (dist_o > band).sum() >= k))
            GATE[ds]['models'][n] = dict(dist=float(dist.mean()), improvement=float(m), ci95=[float(lo), float(hi)],
                                         phi=float(phi[0]), phi_ci95=[float(phi[1]), float(phi[2])], utility=float(u[n].mean()),
                                         criteria=c, passed=all(c.values()))
    json.dump(GATE, open(GATE_PATH, 'w'), indent=2); log('FP32 gate saved.')
for ds, g in GATE.items():
    print(f"\n=== {ds}: band_FP32 {g['band_fp32']:.4f} (descriptive tolerance) | dist(Original) {g['dist_original']:.3f} ===")
    for n, r in g['models'].items():
        print(f"  {n:12s} dist {r['dist']:.4f} | forgetting achieved phi {r['phi']:.3f} (CI {r['phi_ci95'][0]:.3f} to {r['phi_ci95'][1]:.3f}) "
              f"| utility {r['utility']:.3f} -> {'PASSED' if r['passed'] else 'not passed'}")

## 8. Quantization phase (copies of the saved checkpoints)

In [ ]:
CONFIGS = list(itertools.product(CFG['bits'], CFG['schemes'], CFG['scale_modes']))

def quant_seed(ds, seed):
    f = f'{OUT_DIR}/quant/{ds}_seed{seed}.csv'
    if os.path.exists(f): log(f'{ds} seed {seed}: quantization already done, skipping'); return
    t0 = time.time(); W, sp = load_ck(ds, seed); ctx = make_ctx(ds, sp)
    names = ['Original'] + [f'Retrain-{k}' for k in range(1, CFG['n_retrain'] + 1)] + TEST_NAMES
    fp = {n: measure(ctx, W[n]) for n in names}; base = dict(dataset=ds, seed=seed)
    rows = [dict(**base, model=n, bits=32, scheme='fp32', scale_mode='fp32', condition='fp32', draw=-1, clip_frac=0.0, **fp[n][0]) for n in names]
    for bits, scheme, mode in CONFIGS:
        cfg = dict(bits=bits, scheme=scheme, scale_mode=mode)
        def src(n): return W['Original'] if (mode == 'frozen_original' and not n.startswith('Retrain')) else None
        Q = {n: quantize(W[n], bits, scheme, src(n)) for n in names}
        for n in names:
            Wq, S, clip = Q[n]; m, conf = measure(ctx, Wq, thr_fp32=fp[n][0]['threshold'])
            r = dict(**base, model=n, **cfg, condition='quantized', draw=-1, clip_frac=clip, **m)
            if n in TEST_NAMES:
                d_fp = flat2d(W[n]) - flat2d(W['Original']); d_q = flat2d(Wq) - flat2d(Q['Original'][0])
                s = torch.cat([x.flatten() for x in S if x is not None]); rel = d_fp.abs() / s
                SO = Q['Original'][1]; Wo_q = Q['Original'][0]
                ch = [(torch.round(a / sa) != torch.round(b / sb)) for a, sa, b, sb in zip(Wq, S, Wo_q, SO) if a.dim() == 2]
                r.update(code_change_share=float(torch.cat([c.flatten() for c in ch]).float().mean()),
                         **{f'code_change_share_layer{j+1}': float(c.float().mean()) for j, c in enumerate(ch)})
                r.update(codes_identical_to_Q_original=bool(torch.equal(flat2d(Wq), flat2d(Q['Original'][0]))),
                         upd_norm_fp32=float(d_fp.norm()), upd_norm_ratio=float(d_q.norm() / (d_fp.norm() + 1e-30)), upd_cosine=cos(d_q, d_fp),
                         share_upd_erased=float((d_q == 0).float().mean()), med_upd_over_bin=float(rel.median()),
                         share_upd_above_half_bin=float((rel > 0.5).float().mean()),
                         **node_recovery(sp, conf, fp[n][1], fp['Original'][1]))
            rows.append(r)
        for n in ['Original', 'Retrain-1'] + TEST_NAMES:
            for draw in range(CFG['noise_draws']):
                gen = torch.Generator().manual_seed(9000 + 100 * seed + 10 * draw + bits)
                m, conf = measure(ctx, matched_noise(W[n], Q[n][0], scheme, gen), thr_fp32=fp[n][0]['threshold'])
                rows.append(dict(**base, model=n, **cfg, condition='matched_noise', draw=draw, clip_frac=np.nan, **m))
    pd.DataFrame(rows).to_csv(f, index=False)
    log(f'{ds} seed {seed}: quantization done in {time.time()-t0:.0f}s ({len(rows)} rows)')

for ds in CFG['datasets']:
    for seed in CFG['seeds']: quant_seed(ds, seed)
log('Quantization phase finished.')

## 9. Analysis: forgetting achieved vs reversal fraction; pre-registered questions E1-E3

In [ ]:
k = CFG['min_seeds_agree']; summary = []
for ds in CFG['datasets']:
    QD = pd.concat([pd.read_csv(f'{OUT_DIR}/quant/{ds}_seed{s}.csv') for s in CFG['seeds']]); band_fp = GATE[ds]['band_fp32']
    f32 = QD[QD.condition == 'fp32'].pivot_table(index='seed', columns='model', values='auc')
    acc32 = QD[QD.condition == 'fp32'].pivot_table(index='seed', columns='model', values='test_acc')
    gap_fp_o = f32['Original'] - f32['Retrain-1']; dist_o = gap_fp_o.abs()
    for (bits, scheme, mode), g in QD[QD.condition != 'fp32'].groupby(['bits', 'scheme', 'scale_mode']):
        q = g[g.condition == 'quantized']; nz = g[g.condition == 'matched_noise']
        a = q.pivot_table(index='seed', columns='model', values='auc'); acc = q.pivot_table(index='seed', columns='model', values='test_acc')
        an = nz.pivot_table(index='seed', columns='model', values='auc'); band_q = retrain_band(a); eff = max(band_fp, band_q)
        for n in TEST_NAMES:
            gap_fp = f32[n] - f32['Retrain-1']; dist_fp = gap_fp.abs(); gap_q = a[n] - a['Retrain-1']; dist_q = gap_q.abs()
            denom = gap_fp_o - gap_fp; n_unst = int((denom.abs() < band_fp).sum())
            phi = ci95(1 - dist_fp / dist_o); rho = ci95((gap_q - gap_fp) / denom)
            d_dist = dist_q - dist_fp; dd = ci95(d_dist); excess = d_dist - ((an[n] - an['Retrain-1']).abs() - dist_fp); ex = ci95(excess)
            qf = q[q.model == n].set_index('seed'); raw = ci95(qf.raw_shift_D_minus_Heval)
            drops = {mm: float(acc32[mm].mean() - acc[mm].mean()) for mm in ['Original', 'Retrain-1', n]}
            gated = GATE[ds]['models'][n]['passed']; dist_q_o = (a['Original'] - a['Retrain-1']).abs()
            if not gated: verdict = 'ungated (report rho only)'
            elif max(drops.values()) > CFG['collapse_drop']: verdict = 'NOT INTERPRETABLE: model degraded'
            elif not dist_q_o.mean() > eff: verdict = 'NOT INTERPRETABLE: attack signal lost'
            elif dist_q.mean() > eff and dd[1] > 0 and (d_dist > 0).sum() >= k and ex[1] > 0 and (excess > 0).sum() >= k:
                verdict = 'REVERSAL, ' + ('deleted-node-specific shift' if raw[1] > 0 else 'non-specific shift')
            elif dist_q.mean() <= eff and (dist_q <= eff).sum() >= k: verdict = 'PRESERVED'
            else: verdict = 'INCONCLUSIVE'
            if mode == 'frozen_original' and qf.clip_frac.mean() > 0.01: verdict += ' [clipping-dominated]'
            summary.append(dict(dataset=ds, model=n, family='FT' if n == FTN else n[:2], fp32_gate_passed=gated, bits=bits, scheme=scheme,
                scale_mode=mode, verdict=verdict, phi=phi[0], phi_ci_lo=phi[1], phi_ci_hi=phi[2], dist_fp32=dist_fp.mean(), dist_q=dist_q.mean(),
                effective_band=eff, reversal_fraction=rho[0], reversal_ci_lo=rho[1], reversal_ci_hi=rho[2],
                increase_dist=dd[0], increase_ci_lo=dd[1], excess_over_noise=ex[0], excess_ci_lo=ex[1],
                raw_shift_D=qf.raw_shift_D.mean(), raw_shift_Heval=qf.raw_shift_Heval.mean(), raw_diff_ci_lo=raw[1],
                node_recovery_D=qf.node_recovery_D.mean(), node_recovery_Heval=qf.node_recovery_Heval.mean(),
                rho_denominator_mean=float(denom.mean()), rho_denominator_min_abs=float(denom.abs().min()),
                seeds_unstable_rho=f'{n_unst}/{len(denom)}', rho_flagged=bool(n_unst > 0 or phi[0] < 0.2),
                code_change_share=qf.code_change_share.mean(), code_change_share_layer1=qf.code_change_share_layer1.mean(),
                code_change_share_layer2=qf.code_change_share_layer2.mean(),
                acc_drop_max=max(drops.values()), codes_identical_share=qf.codes_identical_to_Q_original.astype(bool).mean(),
                share_upd_erased=qf.share_upd_erased.mean(), med_upd_over_bin=qf.med_upd_over_bin.mean(),
                share_upd_above_half_bin=qf.share_upd_above_half_bin.mean(), upd_cosine=qf.upd_cosine.mean(), clip_frac=qf.clip_frac.mean()))
S = pd.DataFrame(summary); S.to_csv(f'{OUT_DIR}/quantization_summary.csv', index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
P = S[(S.bits == CFG['cell_bits']) & (S.scheme == 'per_channel') & (S.scale_mode == 'recomputed')]
OUT = {}
for ds in CFG['datasets']:
    p = P[P.dataset == ds].set_index('model').loc[TEST_NAMES]
    print(f'\n=== {ds}: INT4 per-channel, recomputed scales ===')
    display(p[['fp32_gate_passed', 'phi', 'phi_ci_lo', 'phi_ci_hi', 'reversal_fraction', 'reversal_ci_lo', 'reversal_ci_hi', 'rho_flagged',
               'seeds_unstable_rho', 'verdict', 'raw_shift_D', 'raw_shift_Heval', 'share_upd_above_half_bin', 'code_change_share',
               'code_change_share_layer1', 'code_change_share_layer2']].round(3))
    e1 = p[(p.index != FTN) & p.fp32_gate_passed & ~p.rho_flagged & p.verdict.str.startswith('REVERSAL')
           & (p.reversal_ci_lo > CFG['existence_min_reversal'])]
    # per-seed phi and rho for the seed-level and cluster-bootstrap analyses (INT4 per-channel recomputed)
    QD = pd.concat([pd.read_csv(f'{OUT_DIR}/quant/{ds}_seed{s}.csv') for s in CFG['seeds']])
    f32 = QD[QD.condition == 'fp32'].pivot_table(index='seed', columns='model', values='auc')
    q4 = QD[(QD.condition == 'quantized') & (QD.bits == CFG['cell_bits']) & (QD.scheme == 'per_channel') & (QD.scale_mode == 'recomputed')
            ].pivot_table(index='seed', columns='model', values='auc')
    d_o = (f32['Original'] - f32['Retrain-1']).abs(); g_o = f32['Original'] - f32['Retrain-1']
    PHI = pd.DataFrame({n: 1 - (f32[n] - f32['Retrain-1']).abs() / d_o for n in VARIANTS})
    RHO = pd.DataFrame({n: ((q4[n] - q4['Retrain-1']) - (f32[n] - f32['Retrain-1'])) / (g_o - (f32[n] - f32['Retrain-1'])) for n in VARIANTS})
    E2 = {}
    for fam, members in [('RC', RC), ('TF', TF)]:
        el = [n for n in members if not p.loc[n, 'rho_flagged']]
        if len(el) < 3: E2[fam] = dict(eligible=el, note='fewer than 3 eligible variants: not computed'); continue
        desc = stats.spearmanr(p.loc[el, 'phi'], p.loc[el, 'reversal_fraction']).correlation
        per_seed = [stats.spearmanr(PHI.loc[s, el], RHO.loc[s, el]).correlation for s in PHI.index]
        rng = np.random.default_rng(12345); boots = []
        for _ in range(2000):
            idx = rng.choice(PHI.index, len(PHI.index), replace=True)
            boots.append(stats.spearmanr(PHI.loc[idx, el].mean(), RHO.loc[idx, el].mean()).correlation)
        boots = np.array(boots); boots = boots[~np.isnan(boots)]
        E2[fam] = dict(eligible=el, spearman_descriptive=float(desc), per_seed=[float(x) for x in per_seed],
                       per_seed_mean=float(np.nanmean(per_seed)), seeds_negative=f'{int(np.sum(np.array(per_seed) < 0))}/{len(per_seed)}',
                       cluster_bootstrap_95=[float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))] if len(boots) else None)
    tfp = [n for n in TF if p.loc[n, 'fp32_gate_passed']]
    OUT[ds] = dict(
        E1='yes: fragile complete forgetting exists (' + ', '.join(e1.index) + ')' if len(e1) else 'no variant both reached Retrain and was undone by INT4',
        E2=E2,
        E3=dict(smallest_free_share_passing=tfp[0] if tfp else None,
                its_reversal_fraction=None if not tfp else [float(p.loc[tfp[0], c]) for c in ['reversal_fraction', 'reversal_ci_lo', 'reversal_ci_hi']],
                its_rho_flagged=None if not tfp else bool(p.loc[tfp[0], 'rho_flagged'])))
    print(f"E1: {OUT[ds]['E1']}")
    for fam, r in E2.items(): print(f"E2 {fam}: {r}")
    print(f"E3: {OUT[ds]['E3']}")
json.dump(OUT, open(f'{OUT_DIR}/questions_outcome.json', 'w'), indent=2, default=str); log('Analysis saved.')

## 10. The central figure: forgetting achieved (FP32) vs reversal after INT4

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(CFG['datasets']), figsize=(7 * len(CFG['datasets']), 5.5))
axes = np.atleast_1d(axes)
for ax, ds in zip(axes, CFG['datasets']):
    p = P[P.dataset == ds].set_index('model')
    for n in TEST_NAMES:
        r = p.loc[n]; col = 'k' if n == FTN else ('C0' if n.startswith('RC') else 'C3')
        ax.errorbar(r.phi, r.reversal_fraction, xerr=[[r.phi - r.phi_ci_lo], [r.phi_ci_hi - r.phi]],
                    yerr=[[r.reversal_fraction - r.reversal_ci_lo], [r.reversal_ci_hi - r.reversal_fraction]],
                    fmt='o' if r.fp32_gate_passed else 's', mfc=col if r.fp32_gate_passed else 'white', color=col, capsize=3)
        ax.annotate(n.replace(' dose=', '@'), (r.phi, r.reversal_fraction), fontsize=7, xytext=(4, 4), textcoords='offset points')
    ax.axhline(0, c='k', lw=0.5); ax.axvline(1, c='grey', ls=':', lw=0.8)
    ax.axvspan(1 - GATE[ds]['band_fp32'] / GATE[ds]['dist_original'], 1.05, color='grey', alpha=0.12, label='within Retrain band')
    ax.set_xlabel('forgetting achieved in FP32 (1 = matches Retrain)'); ax.set_ylabel('reversal fraction after INT4 per-channel')
    ax.set_title(f'{ds}: blue = random confinement, red = targeted freeing, black = FT\n(filled = passed FP32 gate)', fontsize=9)
    ax.legend(fontsize=7, loc='upper right')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/fig_tradeoff.png', dpi=180); plt.show()

## 11. Export everything as a ZIP

In [ ]:
def export():
    base = os.path.basename(OUT_DIR.rstrip('/'))
    shutil.make_archive(base, 'zip', OUT_DIR)
    log(f'ZIP created: {base}.zip ({os.path.getsize(base + ".zip")/1e6:.1f} MB)')
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        from IPython.display import FileLink, display
        print('Kaggle: click the link below, or use the Output panel. In a committed version the ZIP stays in the Output tab.')
        display(FileLink(f'{base}.zip')); return
    try:
        from google.colab import files; files.download(f'{base}.zip')
    except Exception:
        print('Download the ZIP from the file panel on the left.')
export()

### Emergency export
If the session is about to disconnect, run only the cell below.

In [ ]:
export()